In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
base_dir = '../input/competitions/house-prices-advanced-regression-techniques/'
train_df = pd.read_csv(base_dir + 'train.csv')
test_df = pd.read_csv(base_dir + 'test.csv')

In [ ]:
train_df['MSSubClass'] = train_df['MSSubClass'].astype('str')
test_df['MSSubClass'] = test_df['MSSubClass'].astype('str')

**EDA**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train_df['SalePrice'].hist(ax=axes[0], bins=50, color='steelblue')
axes[0].set_title('SalePrice Distribution')
axes[0].set_xlabel('SalePrice')

np.log1p(train_df['SalePrice']).hist(ax=axes[1], bins=50, color='steelblue')
axes[1].set_title('SalePrice Distribution — After Log Transform')
axes[1].set_xlabel('log(SalePrice)')

plt.tight_layout()
plt.show()

print(f"Mean:     {train_df['SalePrice'].mean():,.0f}")
print(f"Median:   {train_df['SalePrice'].median():,.0f}")
print(f"Skewness: {train_df['SalePrice'].skew():.2f}")

In [ ]:
# Top 15 numerical features correlated with SalePrice
numer_cols = train_df.select_dtypes(include=['int64', 'float64']).columns
corr = train_df[numer_cols].corr()['SalePrice'].abs().sort_values(ascending=False)

print(corr.head(15))

# Heatmap of top 10 correlated features
top_features = corr.head(11).index  # includes SalePrice itself
plt.figure(figsize=(10, 8))
sns.heatmap(train_df[top_features].corr(), 
            annot=True, fmt='.2f', cmap='coolwarm',
            square=True)
plt.title('Correlation Matrix — Top 10 Features')
plt.tight_layout()
plt.show()

In [ ]:
corr = train_df[numer_cols].corr()['SalePrice'].abs().sort_values(ascending=False)
top_10_features = corr.head(10).index.tolist()[1:]

fig, axes = plt.subplots(3, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(top_10_features):
    axes[i].scatter(train_df[col], train_df['SalePrice'], 
                    alpha=0.3, color='steelblue')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('SalePrice')
    axes[i].set_title(f'{col} vs SalePrice')

plt.tight_layout()
plt.show()

In [ ]:
# GrLivArea outliers — two famous outliers in this dataset
plt.figure(figsize=(10, 6))
plt.scatter(train_df['GrLivArea'], train_df['SalePrice'], alpha=0.3)
plt.xlabel('GrLivArea')
plt.ylabel('SalePrice')
plt.title('GrLivArea vs SalePrice — Outlier Check')

# Highlight potential outliers
outliers = train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 200000)]
plt.scatter(outliers['GrLivArea'], outliers['SalePrice'], 
            color='red', s=100, label='Suspicious outliers')
outlier_index = outliers.index
plt.legend()
plt.show()

print(f"Suspicious outliers found: {len(outliers)}")
print(outliers[['GrLivArea', 'SalePrice', 'OverallQual', 'Neighborhood']])

In [ ]:
outlier_index

In [ ]:
cater_cols = train_df.select_dtypes(include=['object']).columns.tolist()
print(len(cater_cols))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(cater_cols[:4]):
    order = train_df.groupby(col)['SalePrice'].median().sort_values(ascending=False).index
    sns.boxplot(data=train_df, x=col, y='SalePrice', 
                order=order, ax=axes[i])
    axes[i].set_title(f'{col} vs SalePrice')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sale price over years
train_df.groupby('YrSold')['SalePrice'].median().plot(ax=axes[0], marker='o')
axes[0].set_title('Median SalePrice by Year Sold')
axes[0].set_xlabel('Year')

# Houses built over time
train_df.groupby('YearBuilt')['SalePrice'].median().plot(ax=axes[1])
axes[1].set_title('Median SalePrice by Year Built')
axes[1].set_xlabel('Year Built')

plt.tight_layout()
plt.show()

**Data PreProcessing** 

In [ ]:
test_ids = test_df['Id']
y = np.log1p(train_df['SalePrice'])

train_df = train_df.drop('SalePrice', axis=1)
total_df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

In [ ]:
num_cols = total_df.select_dtypes(include=['float64', 'int64']).columns.tolist()
cat_cols = total_df.select_dtypes(include=['object']).columns.tolist()

print(f"number of numerical columns: {len(num_cols)}")
print(f"number of categorical columns: {len(cat_cols)}")

**Ordinal categories (have some meaningful order):** [LandSlope, ExterQual, ExterCond, BsmtQual, BsmtCond, BsmtExposure, BsmtFinType1, BsmtFinType2, HeatingQC, KitchenQual, FireplaceQu, GarageQual, GarageCond, GarageFinish, PoolQC] 

**Nominal categories (no meaningful order):** [MSSubClass, MSZoning, Street, Alley, LotShape, LandContour, Utilities, LotConfig, Neighborhood, Condition1, Condition2, BldgType, HouseStyle, RoofStyle, RoofMatl, Exterior1st, Exterior2nd, MasVnrType, Foundation, Heating, CentralAir, Electrical, Functional, GarageType, PavedDrive, Fence, MiscFeature, SaleType, SaleCondition]

In [ ]:
ordinal_cols = ['LandSlope', 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 
                'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 
                'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'GarageFinish', 
                'PoolQC']
nominal_cols = ['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 
                'LotConfig', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 
                'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 
                'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 
                'Functional', 'GarageType', 'PavedDrive', 'Fence', 'MiscFeature', 
                'SaleType', 'SaleCondition']

print(len(ordinal_cols) + len(nominal_cols))

In [ ]:
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
quality_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
                'HeatingQC', 'KitchenQual', 'FireplaceQu',
                'GarageQual', 'GarageCond', 'PoolQC']

for col in quality_cols:
    total_df[col] = total_df[col].fillna('None').map(quality_map)

bsmt_exposure_map = {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}
bsmt_fin_map = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}
garage_fin_map = {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0}
landslope_map = {'Gtl': 1, 'Mod': 2, 'Sev': 3}

total_df['BsmtExposure'] = total_df['BsmtExposure'].fillna('None').map(bsmt_exposure_map)
total_df['BsmtFinType1'] = total_df['BsmtFinType1'].fillna('None').map(bsmt_fin_map)
total_df['BsmtFinType2'] = total_df['BsmtFinType2'].fillna('None').map(bsmt_fin_map)
total_df['GarageFinish'] = total_df['GarageFinish'].fillna('None').map(garage_fin_map)
total_df['LandSlope'] = total_df['LandSlope'].map(landslope_map)

In [ ]:
## columns that have nan values or missing numbers
nan_cols = []
for col in total_df.columns.tolist():
    if total_df[col].isnull().sum()>0:
        nan_cols.append(col)

## category columns that have missing values, but they are not actual missing
num_missing = total_df[cat_cols].isnull().sum()
not_nan_cols = num_missing[num_missing > 0].index.tolist()

In [ ]:
cater_cols = set(total_df.select_dtypes(include=['object']).columns.tolist())

num_miss_cols = []
cat_miss_cols = []
for cols in total_df.columns.tolist():
    if total_df[cols].isnull().sum() > 0:
        if cols in cater_cols:
            cat_miss_cols.append(cols)
        else:
            num_miss_cols.append(cols)

In [ ]:
# with open(base_dir + 'data_description.txt', 'r') as f:
#     content = f.read()
# print(content)

In [ ]:
print(len(cat_miss_cols), len(num_miss_cols))
print(cat_miss_cols)
print(num_miss_cols)

In [ ]:
total_df = total_df.drop('Utilities', axis=1)

In [ ]:
real_mising = ['MSZoning', 'LotFrontage', 'MasVnrArea', 'Electrical', 'GarageYrBlt',
              'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 
               'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea']
NA_missing = ['Alley', 'MasVnrType', 'GarageType', 'Fence', 'MiscFeature', 
              'SaleType', 'Functional']

for col in NA_missing:
    total_df[col] = total_df[col].fillna('None')
total_df['LotFrontage'] = total_df.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median()))
total_df['MSZoning'] = total_df.groupby('Neighborhood')['MSZoning'].transform(
    lambda x: x.fillna(x.mode()[0])
)
total_df['MasVnrArea'] = total_df['MasVnrArea'].fillna(0)
total_df['Electrical'] = total_df['Electrical'].fillna(total_df['Electrical'].mode()[0])
total_df['Exterior1st'] = total_df['Exterior1st'].fillna(total_df['Exterior1st'].mode()[0])
total_df['Exterior2nd'] = total_df['Exterior2nd'].fillna(total_df['Exterior2nd'].mode()[0])
total_df[['GarageYrBlt', 'GarageCars', 'GarageArea']] = total_df[
['GarageYrBlt', 'GarageCars', 'GarageArea']].fillna(0)
total_df[['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
          'BsmtFullBath', 'BsmtHalfBath']] = total_df[
['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
 'BsmtFullBath', 'BsmtHalfBath']].fillna(0)

In [ ]:
total_df.isnull().sum().sum()

In [ ]:
total_df[quality_cols].isnull().sum().sum()

In [ ]:
total_df[ordinal_cols].isnull().sum().sum()

In [ ]:
numer_cols = total_df.select_dtypes(include=['int64', 'float64']).columns.tolist()

total_df[numer_cols].skew()

In [ ]:
col_to_exclude = ['OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
                 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 
                  'KitchenAbvGr', 'GarageYrBlt', 'GarageCars']
cols_to_transform = [col  for col in num_cols if abs(
    total_df[col].skew())>0.8 and col not in col_to_exclude]

In [ ]:
for col in cols_to_transform:
    total_df[col] = np.log1p(total_df[col])

In [ ]:
total_df[cols_to_transform].skew()

In [ ]:
print(total_df['TotalBsmtSF'].describe())
print(f"Zero values: {(total_df['TotalBsmtSF'] == 0).sum()}")

In [ ]:
print(total_df['BsmtUnfSF'].describe())
print(f"Zero values: {(total_df['BsmtUnfSF'] == 0).sum()}")

In [ ]:
print(f"Shape: {total_df.shape}")
print(f"Total nulls: {total_df.isnull().sum().sum()}")

In [ ]:
obj_cols = total_df.select_dtypes(include='object').columns.tolist()
print(f"Object columns remaining: {len(obj_cols)}")
print(obj_cols)

In [ ]:
one_hot_cols = []
freq_enc_cols = []
for col in obj_cols:
    print(f"{col}: {train_df[col].nunique()} unique values")
    if train_df[col].nunique() <= 5:
        one_hot_cols.append(col)
    else:
        freq_enc_cols.append(col)

In [ ]:
print(len(train_df), len(test_df), len(train_df) + len(test_df))

In [ ]:
X_features = pd.get_dummies(total_df, columns=one_hot_cols, drop_first=True)
X_features = X_features.drop('Id', axis=1)
for col in freq_enc_cols:
    freq_map = X_features[col].value_counts() / len(total_df)
    X_features[col] = X_features[col].map(freq_map)

In [ ]:
from sklearn.preprocessing import StandardScaler

train_X, test_X = X_features[:len(train_df)].copy(), X_features[len(train_df):].copy()
numerical_columns = X_features.select_dtypes(include=['float64', 'int64']).columns.tolist()
scaler = StandardScaler()
train_X[numerical_columns] = scaler.fit_transform(train_X[numerical_columns])
test_X[numerical_columns] = scaler.transform(test_X[numerical_columns])

In [ ]:
print(train_X[numerical_columns].describe().loc[['mean', 'std']].round(2))
print(test_X[numerical_columns].describe().loc[['mean', 'std']].round(2))

In [ ]:
# No object columns should remain
obj_remaining = X_features.select_dtypes(include='object').columns.tolist()
print(f"Object columns remaining: {obj_remaining}")  # Should be empty

# No nulls
print(f"Total nulls: {X_features.isnull().sum().sum()}")  # Should be 0

# Shape
print(f"Shape: {X_features.shape}")

In [ ]:
print(f"train_X: {train_X.shape}")
print(f"test_X:  {test_X.shape}")
print(f"y:       {y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    train_X, y, 
    test_size=0.2, 
    random_state=42
)

print(f"X_tr:  {X_tr.shape}")
print(f"X_val: {X_val.shape}")
print(f"y_tr:  {y_tr.shape}")
print(f"y_val: {y_val.shape}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

lr = LinearRegression()
lr.fit(X_tr, y_tr)

# Predict
y_pred_lr = lr.predict(X_val)

# Evaluate
rmse_lr = np.sqrt(mean_squared_error(y_val, y_pred_lr))
print(f"Linear Regression RMSE: {rmse_lr:.4f}")

In [ ]:
from sklearn.linear_model import Ridge, Lasso

# Ridge
ridge = Ridge(alpha=0.01)
ridge.fit(X_tr, y_tr)
y_pred_ridge = ridge.predict(X_val)
rmse_ridge = np.sqrt(mean_squared_error(y_val, y_pred_ridge))
print(f"Ridge RMSE: {rmse_ridge:.4f}")

# Lasso
lasso = Lasso(alpha=0.001)
lasso.fit(X_tr, y_tr)
y_pred_lasso = lasso.predict(X_val)
rmse_lasso = np.sqrt(mean_squared_error(y_val, y_pred_lasso))
print(f"Lasso RMSE: {rmse_lasso:.4f}")

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=50
)

xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=100
)

y_pred_xgb = xgb_model.predict(X_val)
rmse_xgb = np.sqrt(mean_squared_error(y_val, y_pred_xgb))
print(f"XGBoost RMSE: {rmse_xgb:.4f}")

In [ ]:
print("\nModel Comparison:")
print(f"Linear Regression : {rmse_lr:.4f}")
print(f"Ridge             : {rmse_ridge:.4f}")
print(f"Lasso             : {rmse_lasso:.4f}")
print(f"XGBoost           : {rmse_xgb:.4f}")

In [ ]:
xgb_model_tuned = xgb.XGBRegressor(
    n_estimators=3000,
    learning_rate=0.01,      # lower learning rate
    max_depth=3,             # shallower trees
    min_child_weight=1,
    subsample=0.7,
    colsample_bytree=0.7,
    gamma=0,
    reg_alpha=0.001,
    reg_lambda=1,
    random_state=42,
    early_stopping_rounds=50
)

xgb_model_tuned.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=200
)

y_pred_tuned = xgb_model_tuned.predict(X_val)
rmse_tuned = np.sqrt(mean_squared_error(y_val, y_pred_tuned))
print(f"XGBoost Tuned RMSE: {rmse_tuned:.4f}")

In [ ]:
from sklearn.model_selection import cross_val_score

# Linear Regression CV
lr_scores = cross_val_score(LinearRegression(), train_X, y, 
                             cv=5, scoring='neg_root_mean_squared_error')
print(f"LR CV RMSE: {-lr_scores.mean():.4f} (+/- {lr_scores.std():.4f})")

# XGBoost CV
xgb_base = xgb.XGBRegressor(n_estimators=100, learning_rate=0.05,
                              max_depth=4, random_state=42)
xgb_scores = cross_val_score(xgb_base, train_X, y,
                              cv=5, scoring='neg_root_mean_squared_error')
print(f"XGB CV RMSE: {-xgb_scores.mean():.4f} (+/- {xgb_scores.std():.4f})")

In [ ]:
# Predict on test set
test_predictions = xgb_model_tuned.predict(test_X)

# Reverse log transformation
final_predictions = np.expm1(test_predictions)

# Create submission file
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})

submission.to_csv('submission.csv', index=False)
print(f"Submission shape: {submission.shape}")
print(submission.head())

In [ ]:
print(f"Min prediction:  ${final_predictions.min():,.0f}")
print(f"Max prediction:  ${final_predictions.max():,.0f}")
print(f"Mean prediction: ${final_predictions.mean():,.0f}")